In [1]:
!pip install gradio reportlab autopep8 black

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 22.2 MB/s eta 0:00:00


In [2]:
import ast
import gradio as gr
from reportlab.pdfgen import canvas
from datetime import datetime

In [3]:
class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):

                params = []
                for arg in node.args.args:
                    param = {
                        "name": arg.arg,
                        "type": ast.unparse(arg.annotation) if arg.annotation else "Any"
                    }
                    params.append(param)

                return_type = ast.unparse(node.returns) if node.returns else "Any"

                return {
                    "name": node.name,
                    "params": params,
                    "return_type": return_type,
                    "node": node
                }

        return None


    @staticmethod
    def analyze_function_logic(func_node):

        analysis = {
            "loops": False,
            "conditions": False,
            "operations": []
        }

        for node in ast.walk(func_node):

            if isinstance(node, (ast.For, ast.While)):
                analysis["loops"] = True

            if isinstance(node, ast.If):
                analysis["conditions"] = True

            if isinstance(node, ast.BinOp):

                if isinstance(node.op, ast.Add):
                    analysis["operations"].append("addition")

                elif isinstance(node.op, ast.Mult):
                    analysis["operations"].append("multiplication")

                elif isinstance(node.op, ast.Sub):
                    analysis["operations"].append("subtraction")

        desc = "Performs computation"

        if analysis["loops"]:
            desc += " using loops"

        if analysis["conditions"]:
            desc += " with conditional logic"

        if analysis["operations"]:
            desc += " and mathematical operations"

        analysis["description"] = desc

        return analysis

In [4]:
def generate_google_docstring(sig, analysis):

    doc = f'"""{sig["name"]} function.\n\n'
    doc += analysis["description"] + ".\n\n"

    doc += "Args:\n"

    for p in sig["params"]:
        doc += f'    {p["name"]} ({p["type"]}): Input parameter.\n'

    doc += "\nReturns:\n"
    doc += f'    {sig["return_type"]}: Result value.\n'

    doc += '"""'

    return doc


def generate_numpy_docstring(sig, analysis):

    doc = f'"""{sig["name"]} function.\n\n'
    doc += analysis["description"] + ".\n\n"

    doc += "Parameters\n----------\n"

    for p in sig["params"]:
        doc += f'{p["name"]} : {p["type"]}\n'
        doc += "    Input parameter\n"

    doc += "\nReturns\n-------\n"
    doc += f'{sig["return_type"]}\n'
    doc += "    Result value\n"

    doc += '"""'

    return doc

In [5]:
def generate_doc(code, style):

    sig = DocGenieAnalyzer.extract_function_signature(code)

    if not sig:
        return "No function detected"

    analysis = DocGenieAnalyzer.analyze_function_logic(sig["node"])

    if style == "google":
        doc = generate_google_docstring(sig, analysis)
    else:
        doc = generate_numpy_docstring(sig, analysis)

    result = code.replace(":", ":\n    " + doc, 1)

    return result

In [6]:
def export_txt(text):

    filename = "docgenie_output.txt"

    with open(filename, "w") as f:
        f.write(text)

    return filename


def export_pdf(text):

    filename = "docgenie_output.pdf"

    c = canvas.Canvas(filename)

    y = 800
    for line in text.split("\n"):
        c.drawString(50, y, line)
        y -= 15

    c.drawString(50, 820, f"Generated on {datetime.now()}")
    c.save()

    return filename

In [7]:
with gr.Blocks() as demo:

    gr.Markdown("# Doc-Genie Python Docstring Generator")

    code_input = gr.Code(
        language="python",
        value="""def factorial(n):

    if n==0:
        return 1

    return n*factorial(n-1)
"""
    )

    style = gr.Radio(["google","numpy"],value="google")

    generate = gr.Button("Generate Docstring")

    output = gr.Code(language="python")

    txt_btn = gr.Button("Download TXT")
    pdf_btn = gr.Button("Download PDF")

    txt_file = gr.File()
    pdf_file = gr.File()

    generate.click(generate_doc,[code_input,style],output)

    txt_btn.click(export_txt,output,txt_file)
    pdf_btn.click(export_pdf,output,pdf_file)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99993c9ff1baef368d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
